# DGS Model Interpretation

This notebook introduces model-interpretation workflows in DeepGeSeq (DGS), focusing on nucleotide-resolution attribution and motif discovery with DeepLIFT, DeepSHAP-style attribution, and TF-MoDISco.

Interpretation is not a replacement for biological validation. It is a way to ask what sequence positions and motifs a trained model appears to rely on for a chosen output task.


## Tutorial series map

This notebook is part of the `Tutorials/0_DGS_usage_models` sequence:
- `0_DGS_data.ipynb`: data, IO, intervals, targets, and loaders
- `1_DGS_models.ipynb`: binary classification and regression scalar models
- `2_DGS_profile_models.ipynb`: profile/count and long-sequence track models
- `3_DGS_gLMs.ipynb`: genomic language model adapters and downstream heads
- **`4_DGS_explain.ipynb`: interpretation methods, attribution artifacts, and motif discovery**

The notebooks are designed to be read in order, but each one keeps its examples self-contained and guarded against missing optional dependencies.


## Audience, prerequisites, and learning goals

This notebook is for users who have a trained or toy DGS-compatible PyTorch model and want to inspect sequence-level evidence for a selected model output.

Prerequisites:

- One-hot DNA tensors and DGS shape conventions.
- Basic PyTorch model inference.
- Familiarity with binary/regression model outputs is helpful.

By the end, you should be able to:

- Explain what nucleotide-resolution attributions measure.
- Choose between DGS `deeplift_shap`, Captum `deeplift`, and Captum `integrated_gradients` methods.
- Save attribution artifacts with the DGS `(N, 4, L)` convention.
- Understand how TF-MoDISco consumes one-hot sequences and attribution scores.
- Run motif-discovery templates only when optional dependencies and real model outputs are available.


## Outline

1. Set up a safe interpretation environment.
2. Compare DeepLIFT, DeepSHAP, and TF-MoDISco roles.
3. Build a tiny motif-scoring model and synthetic one-hot sequences.
4. Run Captum DeepLIFT when available.
5. Run DeepLIFT/SHAP-style attribution when `tangermeme` is available.
6. Compute attribution over a dataset and save artifacts.
7. Prepare TF-MoDISco inputs and use the DGS `motif_enrich` template.
8. Review exercises, pitfalls, and extensions.


## Series-level conventions

Across these notebooks:

- Core examples use tiny synthetic data or local fixtures and avoid model downloads.
- Optional real-data, BigWig, Keras checkpoint, Hugging Face, Evo2, and motif-discovery cells are disabled with `RUN_*` flags until you edit paths and opt in.
- Loader sequence batches are usually `(batch, length, 4)`; CNN-style models usually expect `(batch, 4, length)` after `transpose(1, 2)`.
- Attribution arrays in DGS use `(batch, 4, sequence_length)`.
- Environment guards print skip messages when optional packages such as `torch`, `captum`, `tangermeme`, or the `modisco` CLI are unavailable.


## 1. Environment setup

DGS interpretation has optional dependencies:

- `captum`: Captum DeepLIFT and Integrated Gradients.
- `tangermeme`: legacy DeepLIFT/SHAP-style attribution and seqlet helpers.
- `modisco`: TF-MoDISco-lite command line tool for motif discovery.
- `h5py`: optional HDF5 artifact writing.

Install the explain extra in a clean environment before running real workflows:

```bash
pip install -e ".[explain]"
```


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import shutil
import sys

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "DGS").exists():
            return candidate
    return start


def importable_package(name):
    if importlib.util.find_spec(name) is None:
        return False, "not installed"
    try:
        importlib.import_module(name)
        return True, None
    except Exception as exc:
        # Remove partially imported modules from long-lived notebook kernels.
        for module_name in list(sys.modules):
            if module_name == name or module_name.startswith(name + "."):
                sys.modules.pop(module_name, None)
        return False, f"{type(exc).__name__}: {exc}"

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import numpy as np
    NUMPY_READY = True
except Exception as exc:
    np = None
    NUMPY_READY = False
    NUMPY_IMPORT_ERROR = repr(exc)

TORCH_AVAILABLE, TORCH_IMPORT_ERROR = importable_package("torch")
if TORCH_AVAILABLE:
    TORCH_AVAILABLE, TORCH_NN_IMPORT_ERROR = importable_package("torch.nn")
    TORCH_IMPORT_ERROR = TORCH_NN_IMPORT_ERROR

CAPTUM_AVAILABLE, CAPTUM_IMPORT_ERROR = importable_package("captum")
TANGERMEME_AVAILABLE, TANGERMEME_IMPORT_ERROR = importable_package("tangermeme")
MODISCO_AVAILABLE = shutil.which("modisco") is not None

EXPLAIN_READY = False
if TORCH_AVAILABLE and NUMPY_READY:
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import TensorDataset
        from DGS.DL.Explain import (
            calculate_attributions,
            calculate_attributions_on_ds,
            motif_enrich,
            save_attribution_artifacts,
        )
        EXPLAIN_READY = True
    except Exception as exc:
        EXPLAIN_IMPORT_ERROR = repr(exc)

print(f"Repository root: {REPO_ROOT}")
print(f"numpy ready: {NUMPY_READY}")
if not NUMPY_READY:
    print(f"numpy import error: {NUMPY_IMPORT_ERROR}")
print(f"torch available: {TORCH_AVAILABLE}")
if TORCH_IMPORT_ERROR:
    print(f"torch import issue: {TORCH_IMPORT_ERROR}")
print(f"captum available: {CAPTUM_AVAILABLE}")
if CAPTUM_IMPORT_ERROR:
    print(f"captum import issue: {CAPTUM_IMPORT_ERROR}")
print(f"tangermeme available: {TANGERMEME_AVAILABLE}")
if TANGERMEME_IMPORT_ERROR:
    print(f"tangermeme import issue: {TANGERMEME_IMPORT_ERROR}")
print(f"modisco CLI available: {MODISCO_AVAILABLE}")
print(f"DGS explain API ready: {EXPLAIN_READY}")
if TORCH_AVAILABLE and NUMPY_READY and not EXPLAIN_READY:
    print(f"DGS explain import error: {EXPLAIN_IMPORT_ERROR}")


## 2. Method roles: attribution versus motif discovery

These names are often mentioned together, but they solve different parts of the interpretation workflow.

| Method | DGS method name | Main dependency | Output | Typical use |
|---|---|---|---|---|
| DeepLIFT | `deeplift` | `captum` | attribution scores per base | Fast local importance with a baseline |
| DeepSHAP-style DeepLIFT | `deeplift_shap` | `tangermeme` | attribution scores per base | Genomics-style DeepLIFT/SHAP workflow |
| Integrated Gradients | `integrated_gradients` or `ig` | `captum` | attribution scores per base | Baseline path integral alternative |
| TF-MoDISco-lite | via `motif_enrich` / `modisco` CLI | `modisco` plus attribution inputs | seqlets, patterns, motif report | Discover recurring motifs from many attribution maps |

The usual workflow is: train or load a model, choose a target output, compute attributions for many sequences, save artifacts, then run TF-MoDISco to summarize recurring high-importance patterns.


In [ ]:
method_table = [
    ("deeplift", "Captum DeepLift", "captum", "single baseline, usually zero"),
    ("deeplift_shap", "DeepLIFT/SHAP-style", "tangermeme", "genomics attribution helper"),
    ("integrated_gradients", "Captum Integrated Gradients", "captum", "baseline-to-input path"),
]

print(f"{'DGS method':<22} {'Interpretation':<30} {'Dependency':<14} Notes")
print("-" * 95)
for row in method_table:
    print(f"{row[0]:<22} {row[1]:<30} {row[2]:<14} {row[3]}")


## 3. Build a tiny motif-scoring model

A real interpretation workflow uses a trained model. For a fast tutorial, we build a deterministic toy model that assigns high score to a fixed A-C-G-T pattern at four positions.

This makes it easy to understand what a successful attribution map should highlight.


In [ ]:
if EXPLAIN_READY:
    torch.manual_seed(13)

    sequence_length = 16
    n_examples = 6
    motif_start = 5
    motif_bases = [0, 1, 2, 3]  # A, C, G, T

    base_indices = torch.randint(0, 4, (n_examples, sequence_length))
    for row in range(n_examples):
        base_indices[row, motif_start:motif_start + len(motif_bases)] = torch.tensor(motif_bases)

    X_nlc = torch.nn.functional.one_hot(base_indices, num_classes=4).float()
    X_ncl = X_nlc.transpose(1, 2).contiguous()

    class FixedMotifScoreModel(nn.Module):
        """Linear toy model with known nucleotide-position contributions."""
        def __init__(self, length, motif_start):
            super().__init__()
            weights = torch.zeros(1, 4, length)
            for offset, channel in enumerate([0, 1, 2, 3]):
                weights[0, channel, motif_start + offset] = 1.0
            self.register_buffer("weights", weights)

        def forward(self, x):
            if x.ndim == 3 and x.shape[-1] == 4:
                x = x.transpose(1, 2).contiguous()
            score = (x * self.weights).sum(dim=(1, 2))
            background_task = x[:, 0, :].mean(dim=1)
            return torch.stack([score, background_task], dim=1)

    model = FixedMotifScoreModel(sequence_length, motif_start)
    model.eval()

    with torch.no_grad():
        predictions = model(X_ncl)

    print("X_nlc:", tuple(X_nlc.shape), "= (batch, length, 4)")
    print("X_ncl:", tuple(X_ncl.shape), "= (batch, 4, length)")
    print("predictions:", tuple(predictions.shape), "= (batch, tasks)")
    print("task 0 scores:", predictions[:, 0].tolist())
else:
    print("Skipped: DGS explain API is not ready in this kernel.")


In [ ]:
if EXPLAIN_READY:
    base_names = np.array(["A", "C", "G", "T"])
    first_sequence = "".join(base_names[base_indices[0].numpy()])
    expected_importance = (X_ncl * model.weights).detach().cpu().numpy()
    positional_score = expected_importance[0].sum(axis=0)

    print("first sequence:", first_sequence)
    print("expected motif window:", first_sequence[motif_start:motif_start + 4])
    print("expected important positions:", np.nonzero(positional_score)[0].tolist())
else:
    print("Skipped: DGS explain API is not ready in this kernel.")


## 4. Captum DeepLIFT

DGS exposes Captum DeepLIFT with `method="deeplift"`.

Key choices:

- `target`: output task index to explain. Use `0` for the first output column.
- `baseline`: DGS currently supports zero baselines for Captum methods.
- Input can be `(N, 4, L)` or `(N, L, 4)`; DGS normalizes to `(N, 4, L)`.


In [ ]:
if EXPLAIN_READY and CAPTUM_AVAILABLE:
    deeplift_attr = calculate_attributions(
        model,
        X_ncl,
        target=0,
        device=torch.device("cpu"),
        method="deeplift",
        baseline="zero",
    )

    print("DeepLIFT attribution shape:", deeplift_attr.shape)
    print("top positions in example 0:", np.argsort(-np.abs(deeplift_attr[0]).sum(axis=0))[:6].tolist())
elif EXPLAIN_READY:
    print("Skipped Captum DeepLIFT: install captum or `pip install -e \".[explain]\"`.")
else:
    print("Skipped: DGS explain API is not ready in this kernel.")


## 5. DeepSHAP-style attribution with `deeplift_shap`

DGS uses `method="deeplift_shap"` for the legacy tangermeme DeepLIFT/SHAP-style helper.

This method is common in genomics interpretation workflows, but it requires `tangermeme`. The output is still normalized to the same DGS attribution convention: `(N, 4, L)`.


In [ ]:
if EXPLAIN_READY and TANGERMEME_AVAILABLE:
    shap_attr = calculate_attributions(
        model,
        X_ncl,
        target=0,
        device=torch.device("cpu"),
        method="deeplift_shap",
    )

    print("DeepLIFT/SHAP-style attribution shape:", shap_attr.shape)
    print("top positions in example 0:", np.argsort(-np.abs(shap_attr[0]).sum(axis=0))[:6].tolist())
elif EXPLAIN_READY:
    print("Skipped DeepLIFT/SHAP-style attribution: install tangermeme or `pip install -e \".[explain]\"`.")
else:
    print("Skipped: DGS explain API is not ready in this kernel.")


## 6. Dataset-level attribution

`calculate_attributions_on_ds` processes a dataset and returns two arrays:

- `sequences`: `(N, 4, L)` one-hot input sequences.
- `attributions`: `(N, 4, L)` attribution scores.

Use a batch size for faster inference once your model and dependency stack are working.


In [ ]:
if EXPLAIN_READY and CAPTUM_AVAILABLE:
    ds = TensorDataset(X_ncl)
    sequences, attributions = calculate_attributions_on_ds(
        model,
        ds,
        target=0,
        device=torch.device("cpu"),
        batch_size=2,
        method="deeplift",
    )

    print("dataset sequences:", sequences.shape)
    print("dataset attributions:", attributions.shape)
else:
    print("Skipped dataset attribution: requires DGS explain API and captum.")


## 7. Save attribution artifacts

DGS artifacts store both one-hot sequences and attribution arrays using the channel-first convention `(N, 4, L)`. These artifacts are useful for archiving, auditing, and connecting to motif-discovery steps.

The `.npz` format works with the base NumPy stack. The `.h5` format additionally requires `h5py`.


In [ ]:
if EXPLAIN_READY and CAPTUM_AVAILABLE:
    explain_dir = REPO_ROOT / "Tutorials" / "0_DGS_usage_models" / "_explain_tutorial_tmp"
    explain_dir.mkdir(parents=True, exist_ok=True)

    artifact_path = save_attribution_artifacts(
        explain_dir / "toy_deeplift_attributions.npz",
        sequences,
        attributions,
        method="deeplift",
        target=0,
    )

    loaded = np.load(artifact_path)
    print("saved:", artifact_path)
    print("keys:", sorted(loaded.files))
    print("shape convention:", str(loaded["shape_convention"]))
    print("sequences:", loaded["sequences"].shape)
    print("attributions:", loaded["attributions"].shape)
else:
    print("Skipped artifact saving: requires a computed attribution array.")


## 8. Prepare TF-MoDISco inputs

TF-MoDISco is a motif-discovery step that summarizes recurring high-attribution seqlets across many sequences. It does not replace the attribution method; it consumes attribution maps and one-hot sequences.

DGS provides `motif_enrich`, which calculates attributions, writes TF-MoDISco-lite input files, runs `modisco motifs`, and then runs `modisco report`.

This cell is disabled by default because it requires real attribution dependencies and the `modisco` executable.


In [ ]:
RUN_TFMODISCO_TEMPLATE = False

if RUN_TFMODISCO_TEMPLATE:
    if not EXPLAIN_READY:
        raise RuntimeError("DGS explain API is not ready in this kernel.")
    if not MODISCO_AVAILABLE:
        raise RuntimeError("Install TF-MoDISco-lite and ensure `modisco` is in PATH.")

    ds = TensorDataset(X_ncl)
    motifs_file = motif_enrich(
        model,
        ds,
        target=0,
        output_dir=str(REPO_ROOT / "Tutorials" / "0_DGS_usage_models" / "_explain_tutorial_tmp" / "motif_results"),
        max_seqlets=200,
        device=torch.device("cpu"),
        batch_size=2,
        method="deeplift",
    )
    print("motifs file:", motifs_file)
else:
    print("TF-MoDISco template is disabled. Set RUN_TFMODISCO_TEMPLATE = True after installing dependencies.")


### Manual TF-MoDISco-lite command pattern

When you already have one-hot arrays and attribution arrays, the conceptual command sequence is:

```bash
modisco motifs -s ohe.npz -a shap.npz -n 2000 -o modisco_results.h5
modisco report -i modisco_results.h5 -o motif_report -s motif_report
```

In DGS `motif_enrich`, `ohe.npz` stores one-hot sequences and `shap.npz` stores attribution scores. For real datasets, use enough sequences for stable motif discovery; a tiny toy dataset is only useful for debugging the plumbing.


## 9. Choosing an interpretation workflow

A conservative workflow is:

1. Train or load a model and verify held-out performance.
2. Choose the exact output task to explain.
3. Start with a small batch and `method="deeplift"` if Captum is available.
4. Save attribution artifacts and inspect top positions on known positive examples.
5. Scale to many sequences.
6. Run TF-MoDISco only after the attribution maps look sensible.
7. Treat discovered motifs as hypotheses for follow-up comparison and validation.

For profile models, first define an objective that turns profile outputs into a scalar target, such as a task-specific count, a windowed profile score, or a selected output bin.


## 10. Exercise

Change the target from task `0` to task `1` in the toy model and compare the top attribution positions.

Checklist:

- Use `target=1` in `calculate_attributions`.
- Confirm the attribution map changes because task 1 is the background A-content task.
- Save the result to a separate artifact path.


In [ ]:
if EXPLAIN_READY and CAPTUM_AVAILABLE:
    task1_attr = calculate_attributions(
        model,
        X_ncl,
        target=1,
        device=torch.device("cpu"),
        method="deeplift",
    )
    task0_top = np.argsort(-np.abs(deeplift_attr[0]).sum(axis=0))[:6]
    task1_top = np.argsort(-np.abs(task1_attr[0]).sum(axis=0))[:6]

    print("task 0 top positions:", task0_top.tolist())
    print("task 1 top positions:", task1_top.tolist())
else:
    print("Skipped exercise execution: requires DGS explain API and captum.")


## Common pitfalls

Pitfall 1: explaining an unvalidated model.

- Interpretation is only meaningful after the model has useful predictive behavior.

Pitfall 2: explaining the wrong target.

- Multi-task models require an explicit `target` index. Always map that index back to the biological task name.

Pitfall 3: mixing channel conventions.

- DGS attribution outputs use `(N, 4, L)`. Some loaders or plotting utilities use `(N, L, 4)`.

Pitfall 4: running TF-MoDISco on too few examples.

- TF-MoDISco summarizes repeated patterns. A toy batch can debug commands, but real motif discovery needs many relevant sequences.

Pitfall 5: treating motif discovery as proof.

- Motifs from attributions are hypotheses. Compare them with known motif databases and validate where possible.


## Optional extensions

- Run `method="integrated_gradients"` and compare top positions with DeepLIFT.
- Use a trained `CNN` from `1_DGS_models.ipynb` and explain positive held-out examples.
- Use `build_supervised_dataloader` from `0_DGS_data.ipynb` to explain real intervals.
- Run `motif_enrich` on a larger sequence set and inspect the TF-MoDISco report.
- Convert top attribution windows into BED intervals for genome-browser review.
